In [1]:
import os
import random
import string
import math

import hail as hl
from ukb_utils import hail_init
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
from ko_utils import ko
from ko_utils import io
from ukb_utils import samples


In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

2022-10-18 11:47:06 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Running on Apache Spark version 3.1.3
SparkUI available at http://compa005.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.102-817f6fb3468f
LOGGING: writing to logs/hail/hail_test_export.log


In [3]:
path = "/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/data/unphased/wes_union_calls/ukb_wes_union_calls_200k_chr21.vcf.bgz"

In [5]:
mt = io.import_table(path, "vcf", calc_info = False)

In [8]:
samples = mt.s.collect()

2022-10-18 11:48:22 Hail: INFO: scanning VCF for sortedness...
2022-10-18 11:52:16 Hail: INFO: Coerced sorted VCF - no additional import work to do


In [11]:
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [45]:
def read_interval_idx(interval_path, idx):
    """ Read a specific interval index from a file
    :param interval_path: path to interval file (assumine one line interval)
    :param idx: interval index
    """
    assert idx >= 0
    with open(interval_path, "r") as infile:
        for i, line in enumerate(infile):
            if i == idx:
                return(line)
    


In [39]:
path = "/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/data/phased/wes_union_calls/prephased/chunks/intervals/intervals_spc1000_chr22.tsv"

In [49]:
path = "/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/data/phased/wes_union_calls/prephased/ukb_eur_wes_union_calls_prephased_200k_chr22.vcf.bgz.tmp"

In [62]:
lines = []
with open(path, "r") as infile:
    for line in infile:
        if line.strip():
            lines += [line.strip()]


In [63]:
lines

['data/phased/wes_union_calls/prephased/chunks/ukb_eur_wes_union_calls_200k_chr22-short/chr22_spc100.1of1994.phased.vcf.gz',
 'data/phased/wes_union_calls/prephased/chunks/ukb_eur_wes_union_calls_200k_chr22-short/chr22_spc100.2of1994.phased.vcf.gz']

In [4]:
fam = samples.get_fam()

2022-09-30 14:10:15 Hail: INFO: Reading table without type imputation
  Loading field 'FID' as type str (user-supplied)
  Loading field 'IID' as type str (user-supplied)
  Loading field 'PAT' as type str (user-supplied)
  Loading field 'MAT' as type str (user-supplied)
  Loading field 'SEX' as type str (user-supplied)
  Loading field 'PHEN' as type str (user-supplied)


In [ ]:
fam.

In [65]:
def exclude_trio_parents(mt):
    '''Filter to children of trio parents in the data
    '''
    import hail as hl

    # filter fam file by input samples
    fam = samples.get_fam()
    fam = fam.filter(hl.is_defined(mt.cols()[fam.IID]))
    fam = fam.filter(hl.literal(["TRIO"]).contains(fam.REL))
    
    # get ID of parents
    pids = []
    pids.extend(fam.PAT.collect())
    pids.extend(fam.MAT.collect())
    pids = [x for x in pids if x != "0"]
    assert len(pids) % 2 == 0
    
    # remove by paternal IDs
    return mt.filter_cols(~hl.literal(pids).contains(mt.s))


In [50]:
input_path = "data/mt/annotated/ukb_eur_wes_union_calls_200k_chr21.mt"

In [51]:
mt = io.import_table(input_path, "mt", calc_info = False)

In [52]:
mt.count()

(99869, 176915)

In [66]:
x = exclude_trio_parents(mt)

2022-09-30 14:46:56 Hail: INFO: Reading table without type imputation
  Loading field 'FID' as type str (user-supplied)
  Loading field 'IID' as type str (user-supplied)
  Loading field 'PAT' as type str (user-supplied)
  Loading field 'MAT' as type str (user-supplied)
  Loading field 'SEX' as type str (user-supplied)
  Loading field 'PHEN' as type str (user-supplied)


479203


2022-09-30 14:46:57 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:46:58 Hail: INFO: Coerced sorted dataset


173671


2022-09-30 14:47:01 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:47:02 Hail: INFO: Coerced sorted dataset


399


2022-09-30 14:47:05 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:47:06 Hail: INFO: Coerced sorted dataset
2022-09-30 14:47:10 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:47:10 Hail: INFO: Coerced sorted dataset


In [85]:
def exclude_parents_by_fam(mt, relation = ["TRIO","DUO"]):
    '''

    : param mt: MatrixTable containing indiviudals to filter
    : param relation: list containing relationships to filter by, for example "TRIO" filters
    excludes parents in trios, whereas ["TRIO", "DUO"] excludes parents in trios or duo pedigrees.
    '''

    import hail as hl

    assert len(relation) > 0
    # filter fam file by input samples
    fam = samples.get_fam()
    fam = fam.filter(hl.is_defined(mt.cols()[fam.IID]))
    fam = fam.filter(hl.literal(relation).contains(fam.REL))

    # get ID of parents
    pids = []
    pids.extend(fam.PAT.collect())
    pids.extend(fam.MAT.collect())
    pids = [x for x in pids if x != "0"]
    assert len(pids) % 2 == 0
    
    # remove by paternal IDs
    mt = mt.filter_cols(~hl.literal(pids).contains(mt.s))
    return mt

In [86]:
x = filter_to_offspring_by_fam(mt, relation = ["TRIO","DUO"])

2022-09-30 14:56:34 Hail: INFO: Reading table without type imputation
  Loading field 'FID' as type str (user-supplied)
  Loading field 'IID' as type str (user-supplied)
  Loading field 'PAT' as type str (user-supplied)
  Loading field 'MAT' as type str (user-supplied)
  Loading field 'SEX' as type str (user-supplied)
  Loading field 'PHEN' as type str (user-supplied)
2022-09-30 14:56:35 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:56:35 Hail: INFO: Coerced sorted dataset
2022-09-30 14:56:39 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:56:39 Hail: INFO: Coerced sorted dataset


In [87]:
mt.count()

(99869, 176915)

In [88]:
x.count()

(99869, 176012)

In [2]:
import pysam

ModuleNotFoundError: No module named 'pysam'